In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
DATA = Path(".")  # adjust to your Lab-1 folder

locations = {
    "Connaught Place": (28.6315, 77.2167),
    "Rohini":           (28.7495, 77.0565),
    "Dwarka":           (28.5921, 77.0460),
    "Saket":            (28.5245, 77.2066),
    "Lajpat Nagar":     (28.5677, 77.2431),
    "Karol Bagh":       (28.6519, 77.1909),
    "Anand Vihar":      (28.6469, 77.3153),
    "Vasant Kunj":      (28.5244, 77.1588),
    "Noida":            (28.5355, 77.3910),
    "Delhi Airport":    (28.5562, 77.1000),
}

In [2]:
all_data = []

for name, (lat, lon) in locations.items():
    print(f"Collecting: {name}")

    weather_params = {
        "latitude": lat, "longitude": lon,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": "Asia/Kolkata",
        "past_days": 30,
        "forecast_days": 1,   # trims extra forecast rows beyond past_days
    }
    try:
        r_weather = requests.get(WEATHER_URL, params=weather_params, timeout=20)
        weather_df = pd.DataFrame(r_weather.json()["hourly"])
    except Exception as e:
        print(f"  Weather failed for {name}: {e}")
        continue
    time.sleep(1)

    aq_params = {
        "latitude": lat, "longitude": lon,
        "hourly": "pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide",
        "timezone": "Asia/Kolkata",
        "past_days": 30,
        "forecast_days": 1,
    }
    try:
        r_aq = requests.get(AQ_URL, params=aq_params, timeout=20)
        aq_df = pd.DataFrame(r_aq.json()["hourly"])
    except Exception as e:
        print(f"  Air quality failed for {name}: {e}")
        continue
    time.sleep(1)

    merged = pd.merge(weather_df, aq_df, on="time", how="inner")
    merged["location"] = name
    merged["latitude"] = lat
    merged["longitude"] = lon
    all_data.append(merged)

print("Done. Locations collected:", len(all_data))

Collecting: Connaught Place
Collecting: Rohini
Collecting: Dwarka
Collecting: Saket
Collecting: Lajpat Nagar
Collecting: Karol Bagh
Collecting: Anand Vihar
Collecting: Vasant Kunj
Collecting: Noida
Collecting: Delhi Airport
Done. Locations collected: 10


In [6]:
numeric_cols = ["temperature", "humidity", "precipitation", "wind_speed",
                 "pm2_5", "pm10", "nitrogen_dioxide", "ozone",
                 "sulphur_dioxide", "carbon_monoxide"]

df[numeric_cols] = (
    df.groupby("location")[numeric_cols]
      .transform(lambda s: s.interpolate(limit_direction="both"))
)

print("Missing after handling:")
print(df.isna().sum())

Missing after handling:
location            0
latitude            0
longitude           0
time                0
temperature         0
humidity            0
precipitation       0
wind_speed          0
pm2_5               0
pm10                0
nitrogen_dioxide    0
ozone               0
sulphur_dioxide     0
carbon_monoxide     0
dtype: int64


In [3]:
df = pd.concat(all_data, ignore_index=True)

df = df.rename(columns={
    "temperature_2m": "temperature",
    "relative_humidity_2m": "humidity",
    "wind_speed_10m": "wind_speed",
})

df = df[["location", "latitude", "longitude", "time",
         "temperature", "humidity", "precipitation", "wind_speed",
         "pm2_5", "pm10", "nitrogen_dioxide", "ozone",
         "sulphur_dioxide", "carbon_monoxide"]]

df["time"] = pd.to_datetime(df["time"])
print("Shape:", df.shape)
df.head()

Shape: (7440, 14)


,location,latitude,longitude,time,temperature,humidity,precipitation,wind_speed,pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide
0,Connaught Place,28.6315,77.2167,2026-07-17 00:00:00,34.2,61,0.0,0.5,255.2,1054.3,91.7,0.0,45.3,638.0
1,Connaught Place,28.6315,77.2167,2026-07-17 01:00:00,33.7,59,0.0,1.2,261.7,1087.8,91.8,0.0,45.9,471.0
2,Connaught Place,28.6315,77.2167,2026-07-17 02:00:00,33.2,59,0.0,0.6,267.3,1113.9,90.9,0.0,46.5,345.0
3,Connaught Place,28.6315,77.2167,2026-07-17 03:00:00,32.1,65,0.0,2.9,271.0,1113.9,90.0,0.0,48.4,279.0
4,Connaught Place,28.6315,77.2167,2026-07-17 04:00:00,31.5,71,0.0,2.7,267.3,1039.4,88.0,0.0,50.3,255.0


In [5]:
df.to_csv(DATA / "delhi_air_quality_weather.csv", index=False)
print("Saved to", DATA / "delhi_air_quality_weather.csv")
print("Total records:", len(df))

Saved to delhi_air_quality_weather.csv
Total records: 7440


In [7]:
print(df.groupby("location").size())

location
Anand Vihar        744
Connaught Place    744
Delhi Airport      744
Dwarka             744
Karol Bagh         744
Lajpat Nagar       744
Noida              744
Rohini             744
Saket              744
Vasant Kunj        744
dtype: int64


In [10]:
# --- Final check: list all the files we collected ---

print("Files collected in this lab:\n")
for f in sorted(DATA.glob("*")):
    print(f"  {f.name:22s}  {f.stat().st_size:>8,} bytes")

Files collected in this lab:

  .ipynb_checkpoints         4,096 bytes
  air_quality.csv           11,215 bytes
  banglore houses exp 1.ipynb     6,538 bytes
  bengaluru_houses.csv       2,446 bytes
  collecting air quality exp 3.ipynb    50,215 bytes
  collecting quotes.ipynb     5,315 bytes
  creating large aq and weather dataset.ipynb    13,114 bytes
  data                       4,096 bytes
  delhi_air_quality_weather.csv   704,898 bytes
  Lab_Data_Collection_Simple.ipynb    75,631 bytes
  quotes.csv                 1,127 bytes
  weather api exp 2.ipynb    14,999 bytes
